In [1]:
import requests
import json
import time
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
from datetime import datetime, timedelta
import numpy as np

In [2]:
gdf = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv")
gdf = gdf.drop(columns = "Unnamed: 0")


#### Hotels

#### Buurt

In [7]:
import shapely
brt = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_BUURT (1).csv", delimiter=';')[["Buurt", "Wijk", "WKT_LNG_LAT"]]
brt = gpd.GeoDataFrame(brt, geometry = brt["WKT_LNG_LAT"].apply(shapely.wkt.loads), crs = "EPSG:4326")
brt = brt.to_crs("EPSG:28992")
brt["centroid"] = brt.centroid
brt["centroid"] = brt["centroid"].to_crs("EPSG:4326")
brt = brt.set_geometry("centroid")
brt["longitude"] = brt.geometry.x
brt["latitude"] = brt.geometry.y

#### Wijk

In [6]:
import shapely
wijk = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/INDELING_WIJK.csv", delimiter=';')[["Wijk", "WKT_LNG_LAT"]]
wijk = gpd.GeoDataFrame(wijk, geometry = wijk["WKT_LNG_LAT"].apply(shapely.wkt.loads), crs = "EPSG:4326")
wijk = wijk.to_crs("EPSG:28992")
wijk["centroid"] = wijk.centroid
wijk["centroid"] = wijk["centroid"].to_crs("EPSG:4326")
wijk = wijk.set_geometry("centroid")
wijk["longitude"] = wijk.geometry.x
wijk["latitude"] = wijk.geometry.y

#### Cores

In [8]:
gdf = gpd.read_file("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/new cores.shp")
gdf["longitude"] = gdf.geometry.x
gdf["latitude"] = gdf.geometry.y

### Script for pedestrian and cycling

In [12]:
# Your API credentials
app_id = 'dd659861' 
api_key = 'faae38d28d3bef5a3bd6f172f3b371a7'  

# API endpoint
url = "https://api.traveltimeapp.com/v4/time-map"

# Request headers
headers = {
    "Content-Type": "application/json",
    "Accept": "application/geo+json",
    "X-Application-Id": app_id,
    "X-Api-Key": api_key
}


# Travel settings
transport_modes = ["walking", "cycling"]
travel_times = [5*60, 15*60, 25*60]  # Convert minutes to seconds

# Add new columns for each isochrone result
for mode in transport_modes:
    for time_mins in [5, 15, 25]:  # Keep in minutes for column names
        col_name = f"{mode}_{time_mins}min"
        gdf[col_name] = None  # Initialize columns
        
# Function to send request to API
def get_isochrone(lat, lng, travel_time, transport_mode):
    payload = {
        "departure_searches": [
            {
                "id": "isochrone",
                "coords": {"lat": lat, "lng": lng},
                "departure_time": "2025-03-29T12:00:00Z",  # Format to ISO 8601
                "travel_time": travel_time,
                "transportation": {"type": transport_mode}
            }
        ]
    }

    response = requests.post(url, headers=headers, data=json.dumps(payload))
    
    if response.status_code == 200:
        geojson_data = response.json()
        features = geojson_data.get("features", [])
        
        if features:
            return shape(features[0]["geometry"])  # Return the geometry
        else:
            return None  # No valid geometry returned
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None  # Return None in case of error

# Loop through hotels and request isochrones
requests_count = 0
for index, row in gdf.iterrows():
    lat, lng = row["latitude"], row["longitude"]

    for mode in transport_modes:
        for travel_time in travel_times:
            col_name = f"{mode}_{travel_time // 60}min"  # Convert to minutes for column name
            geometry = get_isochrone(lat, lng, travel_time, mode)
            gdf.at[index, col_name] = geometry  # Store result in DataFrame

            # Handle API rate limit
            requests_count += 1
            if requests_count >= 5:  # API allows 5 requests per minute
                print("Reached request limit, sleeping for 60 seconds...")
                time.sleep(60)
                requests_count = 0  # Reset counter

Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...


### Script for public transport

In [16]:
# For public transport
# Travel settings
transport_mode = "public_transport"
travel_times = [15 * 60, 30 * 60]  # Convert minutes to seconds

# Define time range (from 12:00 to 12:30 with 5-min gaps)
start_time = datetime(2025, 3, 29, 12, 0)
end_time = datetime(2025, 3, 29, 12, 30)
time_intervals = [start_time + timedelta(minutes=i) for i in range(0, 31, 5)]  # Every 5 minutes

# Add new columns for each time slot
for travel_time in [15, 30]:  # Keep in minutes for column names
    for time_slot in time_intervals:
        col_name = f"{transport_mode}_{travel_time}min_{time_slot.strftime('%H-%M')}"
        gdf[col_name] = None  # Initialize columns

# Function to send request to API
def get_isochrone(lat, lng, travel_time, transport_mode, departure_time):
    payload = {
        "departure_searches": [
            {
                "id": "isochrone",
                "coords": {"lat": lat, "lng": lng},
                "departure_time": departure_time.strftime("%Y-%m-%dT%H:%M:%SZ"),  # Format to ISO 8601
                "travel_time": travel_time,
                "transportation": {"type": transport_mode}
            }
        ]
    }

    response = requests.post(url, headers=headers, data=json.dumps(payload))
    
    if response.status_code == 200:
        geojson_data = response.json()
        features = geojson_data.get("features", [])
        
        if features:
            return shape(features[0]["geometry"])  # Return the geometry
        else:
            return None  # No valid geometry returned
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None  # Return None in case of error

# Loop through hotels and request isochrones
requests_count = 0
for index, row in gdf.iterrows():
    lat, lng = row["latitude"], row["longitude"]

    for travel_time in travel_times:
        for time_slot in time_intervals:
            col_name = f"{transport_mode}_{travel_time//60}min_{time_slot.strftime('%H-%M')}"
            geometry = get_isochrone(lat, lng, travel_time, transport_mode, time_slot)
            gdf.at[index, col_name] = geometry  # Store result in DataFrame

            # Handle API rate limit
            requests_count += 1
            if requests_count >= 5:  # API allows 60 requests per minute
                print("Reached request limit, sleeping for 60 seconds...")
                time.sleep(60)
                requests_count = 0  # Reset counter


Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Reached request limit, sleeping for 60 seconds...
Error 400: {"http_status":400,"error_code":3,"description":"Failed to parse json - syntax error","documentation_link":"https://docs.traveltime.com/reference/error-codes","additional_info":{"syntax_errors":["Non-standard token 'NaN': enable `JsonReadFeature.ALLOW_NON_NUMERIC_NUMBERS` to allow"]}}
Err

KeyboardInterrupt: 

In [153]:
#brt.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/buurt_traveltime.csv")

In [22]:
#wijk.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/wijk_traveltime.csv")

In [ ]:
#hotels.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/hotels_traveltime.csv")

In [24]:
#gdf.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/TravelTime/cores_traveltime.csv")